# Tempo computacional dos métodos no cenário aplicado

A figura compara o tempo da etapa de otimização, sem incluir ajuste das superfícies, construção do payoff, exportação ou geração da figura. Para o C-NBI é usada somente a etapa instrumentada de resolução dos subproblemas; para NSGA-III e MOEA/D, os tempos das dez sementes da execução corrigida; para o VRF-NBI, dez repetições do solver validado do repositório, com três fatores e 66 pesos confirmados por `VRF_Pareto.xlsx`.

A rotina original do VRF-NBI não registra tempo. A reexecução autorizada emprega a formulação validada do repositório e preserva em CSV tanto os tempos quanto os diagnósticos de convergência. As 66 interseções convergiram em todas as repetições.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, LogLocator

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'notebooks').is_dir() and (p / 'configs').is_dir())
CNBI_STAGES = ROOT / 'data' / 'applied' / 'orcamento_computacional_etapas.csv'
EA_MANIFEST = ROOT / 'results' / 'applied' / 'applied_8d_equal_budget' / 'final_manifest.csv'
VRF_TIMES = ROOT / 'results' / 'applied' / 'timing' / 'vrf_nbi_benchmark.csv'
VRF_REFERENCE = ROOT / 'data' / 'applied' / 'VRF_Pareto.xlsx'
VRF_BENCHMARK_SCRIPT = ROOT / 'scripts' / 'benchmark_vrf_nbi_applied.py'
OUT = ROOT / 'results' / 'applied' / 'figures_dissertation'
OUT.mkdir(parents=True, exist_ok=True)

for path in (CNBI_STAGES, EA_MANIFEST, VRF_TIMES, VRF_REFERENCE, VRF_BENCHMARK_SCRIPT):
    if not path.exists():
        raise FileNotFoundError(path)

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 9.5,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

In [ ]:
cnbi = pd.read_csv(CNBI_STAGES)
ea = pd.read_csv(EA_MANIFEST)
vrf = pd.read_csv(VRF_TIMES)

rows = [pd.DataFrame({
    'method': ['C-NBI'],
    'wall_seconds': [float(cnbi.loc[cnbi['chave'].eq('05_otimizacao_cnbi'), 'tempo_parede_s'].iloc[0])],
})]
rows.append(pd.DataFrame({
    'method': 'VRF-NBI',
    'wall_seconds': pd.to_numeric(vrf['wall_seconds']),
}))

ea_method = ea['method'].astype(str).str.upper().replace({'NSGAIII': 'NSGA-III', 'MOEAD': 'MOEA/D'})
rows.append(pd.DataFrame({
    'method': ea_method,
    'wall_seconds': pd.to_numeric(ea['wall_seconds']),
}))
timings = pd.concat(rows, ignore_index=True)
order = ['C-NBI', 'VRF-NBI', 'NSGA-III', 'MOEA/D']
timings['method'] = pd.Categorical(timings['method'], categories=order, ordered=True)
timings = timings.dropna(subset=['method', 'wall_seconds'])

summary = (timings.groupby('method', observed=True)['wall_seconds']
           .agg(n='count', q1=lambda s: s.quantile(0.25), median='median', q3=lambda s: s.quantile(0.75))
           .reindex(order))
summary

In [ ]:
colors = {
    'C-NBI': '#E68613',
    'VRF-NBI': '#2878B5',
    'NSGA-III': '#3A923A',
    'MOEA/D': '#D1495B',
}
markers = {'C-NBI': 'o', 'VRF-NBI': 'D', 'NSGA-III': '^', 'MOEA/D': 's'}

fig, ax = plt.subplots(figsize=(6.30, 3.65))  # aproximadamente 16 cm de largura
x = np.arange(len(order))
for i, method in enumerate(order):
    row = summary.loc[method]
    median = float(row['median'])
    lower = median - float(row['q1'])
    upper = float(row['q3']) - median
    ax.errorbar(
        i, median, yerr=np.array([[lower], [upper]]),
        fmt=markers[method], markersize=8.2 if method == 'C-NBI' else 7.2,
        color=colors[method], markerfacecolor=colors[method],
        markeredgecolor='white', markeredgewidth=0.75,
        ecolor=colors[method], elinewidth=2.0, capsize=5, capthick=1.6,
        zorder=3,
    )
    ax.annotate(
        f'{median:.2f}', (i, median), xytext=(0, 9), textcoords='offset points',
        ha='center', va='bottom', fontsize=8.5, color='#303030',
    )

ax.set_xticks(x, order)
ax.set_ylabel('Tempo de otimização (s)')
ax.set_xlim(-0.55, len(order) - 0.45)
ax.set_yscale('log')
ax.set_ylim(0.035, 30)
ax.yaxis.set_major_locator(LogLocator(base=10, numticks=5))
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{value:g}'))
ax.yaxis.set_minor_locator(LogLocator(base=10, subs=(2, 5)))
ax.grid(axis='y', which='major', color='#D0D0D0', linewidth=0.70, alpha=0.78)
ax.grid(axis='y', which='minor', color='#E5E5E5', linewidth=0.50, alpha=0.55)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#777777')
ax.spines['bottom'].set_color('#777777')
fig.subplots_adjust(left=0.12, right=0.99, top=0.93, bottom=0.18)

png = OUT / 'fig_tempo_metodos_aplicado.png'
pdf = OUT / 'fig_tempo_metodos_aplicado.pdf'
fig.savefig(png, dpi=300, facecolor='white')
fig.savefig(pdf, facecolor='white')
plt.show()
print(f'PNG: {png}')
print(f'PDF: {pdf}')

**Leitura do gráfico.** O ponto representa a mediana e a barra vertical representa o intervalo interquartil. Como há somente uma execução instrumentada do C-NBI, seu intervalo é nulo. O VRF-NBI foi repetido dez vezes e convergiu nos 66 subproblemas em todas elas; NSGA-III e MOEA/D usam as dez sementes da campanha corrigida. O eixo vertical é logarítmico.